
# Customer Churn Analysis & Cost-Optimized Retention Strategy

## Problem Statement

The objective of this project is to identify key drivers of customer churn and develop a predictive model to enable targeted retention strategies.

Beyond traditional evaluation metrics, the analysis incorporates a cost-based approach by assigning higher penalties to missed churners (false negatives). This ensures that model decisions are aligned with real-world business impact, enabling more effective and resource-efficient customer retention.

In [6]:
import pandas as pd 
import numpy as np
import plotly.graph_objects as go
import xgboost as xgb

from scipy.stats import gaussian_kde
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix

import warnings
warnings.filterwarnings("ignore")

In [7]:
df = pd.read_csv("Churn_Modelling.csv")

---
## Data Understanding

The dataset contains customer demographics, account details, and activity indicators used to analyze churn behavior.

The target variable `Exited` represents whether a customer churned (1) or stayed (0). Initial exploration is performed to understand data structure, feature types, and completeness.


In [8]:
df.head(20)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,6,15574012,Chu,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
6,7,15592531,Bartlett,822,France,Male,50,7,0.00,2,1,1,10062.80,0
7,8,15656148,Obinna,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1
8,9,15792365,He,501,France,Male,44,4,142051.07,2,0,1,74940.50,0
9,10,15592389,H?,684,France,Male,27,2,134603.88,1,1,1,71725.73,0


In [9]:
df.tail(20)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
9980,9981,15719276,T'ao,741,Spain,Male,35,6,74371.49,1,0,0,99595.67,0
9981,9982,15672754,Burbidge,498,Germany,Male,42,3,152039.70,1,1,1,53445.17,1
9982,9983,15768163,Griffin,655,Germany,Female,46,7,137145.12,1,1,0,115146.40,1
9983,9984,15656710,Cocci,613,France,Male,40,4,0.00,1,0,0,151325.24,0
9984,9985,15696175,Echezonachukwu,602,Germany,Male,35,7,90602.42,2,1,1,51695.41,0
9985,9986,15586914,Nepean,659,France,Male,36,6,123841.49,2,1,0,96833.00,0
9986,9987,15581736,Bartlett,673,Germany,Male,47,1,183579.54,2,0,1,34047.54,0
9987,9988,15588839,Mancini,606,Spain,Male,30,8,180307.73,2,1,1,1914.41,0
9988,9989,15589329,Pirozzi,775,France,Male,30,4,0.00,2,1,0,49337.84,0
9989,9990,15605622,McMillan,841,Spain,Male,28,4,0.00,2,1,1,179436.60,0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [11]:
df.describe()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [12]:
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


## Data Cleaning

Irrelevant columns such as `RowNumber`, `CustomerId`, and `Surname` are removed as they do not contribute to predicting customer churn.

Removing such identifiers helps reduce noise and prevents the model from learning meaningless patterns.

In [13]:
df.drop(columns = ['RowNumber','CustomerId','Surname'], inplace = True)
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [14]:
df.duplicated().any()

np.False_

---
## Exploratory Data Analysis (EDA)

EDA is performed to identify patterns and relationships between features and customer churn.

The focus is on comparing churn rates across different categories and understanding how numerical features vary between churned and non-churned customers.

These insights help guide feature importance and model expectations.


In [15]:
print(df.groupby('Geography')['Exited'].mean())

print(df.groupby('Gender')['Exited'].mean())

print(df.groupby('HasCrCard')['Exited'].mean())

print(df.groupby('IsActiveMember')['Exited'].mean())

Geography
France     0.161548
Germany    0.324432
Spain      0.166734
Name: Exited, dtype: float64
Gender
Female    0.250715
Male      0.164559
Name: Exited, dtype: float64
HasCrCard
0    0.208149
1    0.201843
Name: Exited, dtype: float64
IsActiveMember
0    0.268509
1    0.142691
Name: Exited, dtype: float64


In [16]:
df.groupby('Exited').agg(
    Age_Mean = ('Age','mean'),
    Age_Median = ('Age','median'),
    Tenure_mean = ('Tenure','mean'),
    Tenure_Median = ('Tenure','median'),
    Balance_Mean = ('Balance','mean'),
    Balance_Median = ('Balance','median'),
    Products_Mean = ('NumOfProducts','mean'),
    Products_Median = ('NumOfProducts','median'),
    EstimatedSalary_mean = ('EstimatedSalary','mean'),
    EstimatedSalary_Median = ('EstimatedSalary','median'),
    CreditScore_Mean = ('CreditScore','mean'),
    CreditScore_Median = ('CreditScore','median') )

,Age_Mean,Age_Median,Tenure_mean,Tenure_Median,Balance_Mean,Balance_Median,Products_Mean,Products_Median,EstimatedSalary_mean,EstimatedSalary_Median,CreditScore_Mean,CreditScore_Median
Exited,,,,,,,,,,,,
0,37.408389,36.0,5.033279,5.0,72745.296779,92072.68,1.544267,2.0,99738.391772,99645.04,651.853196,653.0
1,44.837997,45.0,4.932744,5.0,91108.539337,109349.29,1.475209,1.0,101465.677531,102460.84,645.351497,646.0


In [17]:
# ===============================
# DASHBOARD 1: OVERVIEW
# ===============================

churn_counts = df['Exited'].value_counts()
geo_churn = df.groupby('Geography')['Exited'].mean().sort_values(ascending=False)

active_churn = df.groupby('IsActiveMember')['Exited'].mean()
active_churn.index = ['Inactive', 'Active']

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "domain"}, {"type": "xy"}, {"type": "xy"}]],
    subplot_titles=(
        "Churn Distribution",
        "Churn by Geography",
        "Churn by Activity"
    )
)

# --- Donut ---
fig.add_trace(
    go.Pie(
        labels=['No Churn', 'Churn'],
        values=[churn_counts[0], churn_counts[1]],
        hole=0.5,
        marker=dict(colors=['#1f77b4', '#d62728']),
        textinfo='percent+label',
        pull=[0, 0.05],
        hovertemplate=
            "<b>%{label}</b><br>" +
            "Customers: %{value}<br>" +
            "Percentage: %{percent}<extra></extra>",
        showlegend=False
    ),
    row=1, col=1
)

# --- Geography ---
fig.add_trace(
    go.Bar(
        x=geo_churn.index,
        y=geo_churn.values,
        text=[f"{v*100:.1f}%" for v in geo_churn.values],
        textposition='outside',
        marker=dict(
            color=['#d62728' if v == geo_churn.max() else '#1f77b4' for v in geo_churn.values]
        ),
        hovertemplate=
            "<b>%{x}</b><br>" +
            "Churn Rate: %{y:.1%}<extra></extra>",
        showlegend=False
    ),
    row=1, col=2
)

# --- Activity ---
fig.add_trace(
    go.Bar(
        x=active_churn.index,
        y=active_churn.values,
        text=[f"{v*100:.1f}%" for v in active_churn.values],
        textposition='outside',
        marker=dict(color=['#d62728', '#2ca02c']),
        hovertemplate=
            "<b>%{x}</b><br>" +
            "Churn Rate: %{y:.1%}<extra></extra>",
        showlegend=False
    ),
    row=1, col=3
)

fig.update_layout(
    title="<b>Customer Churn Overview</b>",
    title_x=0.5,
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=13),
    margin=dict(l=40, r=40, t=100, b=40)
)

fig.update_yaxes(tickformat=".0%", row=1, col=2)
fig.update_yaxes(tickformat=".0%", row=1, col=3, automargin=True)

fig.show()


# ===============================
# DASHBOARD 2: BEHAVIOR
# ===============================

# -------------------------------
# Products: mean + count
# -------------------------------
prod_stats = df.groupby('NumOfProducts')['Exited'].agg(['mean', 'count'])

# Relative threshold (1% of dataset)
threshold = 0.01 * len(df)

# Color logic
colors = [
    '#ffb347' if c < threshold else '#ff7f0e'   
    for c in prod_stats['count']
]

# Churn count
churn_counts = (prod_stats['mean'] * prod_stats['count']).astype(int)

# -------------------------------
# Subplots
# -------------------------------
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "Age Distribution by Churn",
        "Balance by Churn",
        "Churn Rate by Products"
    )
)

# -------------------------------
# 1. Age KDE
# -------------------------------
x_range = np.linspace(df['Age'].min(), df['Age'].max(), 300)

for label, color, name in [
    (0, '#1f77b4', 'No Churn'),
    (1, '#d62728', 'Churn')
]:
    kde = gaussian_kde(df[df['Exited'] == label]['Age'])

    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=kde(x_range),
            mode='lines',
            name=name,
            line=dict(width=3, color=color),
            fill='tozeroy',
            opacity=0.3,
            hovertemplate=
                f"<b>{name}</b><br>" +
                "Age: %{x:.0f}<br>" +
                "Density: %{y:.3f}<extra></extra>"
        ),
        row=1, col=1
    )
fig.update_layout(
    legend=dict(
        x=0.23,
        y=1.05
    )
)

# -------------------------------
# 2. Balance Boxplot
# -------------------------------
fig.add_trace(
    go.Box(
        x=df['Exited'],
        y=df['Balance'],
        marker_color='#1f77b4',
        name='Balance',
        hovertemplate=
            "Churn: %{x}<br>" +
            "Balance: %{y:$,.0f}<extra></extra>",
        showlegend=False
    ),
    row=1, col=2
)

# -------------------------------
# 3. Products (FINAL FIX)
# -------------------------------
fig.add_trace(
    go.Bar(
        x=prod_stats.index,
        y=prod_stats['mean'],
        text=[
            f"{rate*100:.1f}%<br>(n={cnt})"
            for rate, cnt in zip(prod_stats['mean'], prod_stats['count'])
        ],
        textposition='outside',
        customdata=np.stack((prod_stats['count'], churn_counts), axis=-1),
        marker=dict(color=colors),
        hovertemplate=
            "<b>Products: %{x}</b><br>" +
            "Churn Rate: %{y:.1%}<br>" +
            "Customers: %{customdata[0]}<br>" +
            "Churned: %{customdata[1]}<extra></extra>",
        showlegend=False
    ),
    row=1, col=3
)


# -------------------------------
# Layout
# -------------------------------
fig.update_layout(
       title=dict(
        text="<b>Customer Behavior & Churn Drivers</b>",
        x=0.5
    ),
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=13),
    margin=dict(l=50, r=50, t=100, b=80)  
)
fig.add_annotation(
    text="<span style='font-size:12px;color:gray'>Product 4 has low sample size (n=60)</span>",
    x=0.92, y=-0.25,
    xref="paper", yref="paper",
    showarrow=False,
    xanchor="right"
)

fig.update_xaxes(title_text="Age", row=1, col=1)
fig.update_yaxes(title_text="Density", row=1, col=1)

fig.update_xaxes(title_text="Churn (0 = No, 1 = Yes)", row=1, col=2)
fig.update_yaxes(title_text="Balance", row=1, col=2)

fig.update_xaxes(title_text="Number of Products", row=1, col=3)
fig.update_yaxes(range=[0, 1.16], title_text="Churn Rate", tickformat=".0%", row=1, col=3, automargin=True)

fig.show()

### Key EDA Insights

The dataset exhibits class imbalance (~20% churn), which is considered during modeling and evaluation.

- Churn is highest in Germany and among inactive customers, making geography and engagement the strongest drivers.
- Churned customers are older on average and tend to have higher account balances.
- Product ownership shows a non-linear relationship: churn is lowest at two products but increases sharply at three, while results for four products are based on a limited sample.
- Tenure, estimated salary, and credit score show minimal influence on churn.

---
## Train-Test Split

The dataset is split into training and testing sets using stratified sampling to preserve the proportion of churned and non-churned customers.

This ensures that model evaluation is reliable and reflects real-world class distribution.

In [18]:
X = df.drop(columns=['Exited'])
y = df["Exited"]


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Encoding Categorical Variables

Categorical variables such as `Gender` and `Geography` are converted into numerical format.

- `Gender` is binary encoded.
- `Geography` is one-hot encoded to avoid introducing artificial ordering.

One category is dropped to prevent multicollinearity, ensuring stable model behavior.

In [20]:
X_train['Gender'] = X_train['Gender'].map({'Male':1,'Female':0})
X_test['Gender'] = X_test['Gender'].map({'Male':1,'Female':0})

In [21]:
X_train = pd.get_dummies(X_train, columns=['Geography'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['Geography'], drop_first=True)

In [22]:
# Ensure train and test have same columns after encoding
X_train, X_test = X_train.align(X_test, join = "left", axis=1, fill_value = 0)

## Feature Scaling

Numerical features are standardized using StandardScaler.

Scaling ensures that features are on a comparable scale, which is particularly important for models like Logistic Regression.

The scaler is fitted only on the training data to prevent data leakage.

In [23]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = [
    'CreditScore', 'Age', 'Tenure', 'Balance',
    'NumOfProducts', 'EstimatedSalary'
]

#Learn mean & std from training data, then scale it
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

#Use training data mean & std to scale test data
X_test[num_cols] = scaler.transform(X_test[num_cols])

---
## Baseline Model: Logistic Regression

Logistic Regression is used as the initial baseline model due to its simplicity and interpretability.

This model helps establish a reference point for evaluating improvements from more complex models and techniques.

In [24]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)


LogisticRegression()

In [25]:
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.808


In [26]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[1540   53]
 [ 331   76]]


In [27]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.97      0.89      1593
           1       0.59      0.19      0.28       407

    accuracy                           0.81      2000
   macro avg       0.71      0.58      0.59      2000
weighted avg       0.78      0.81      0.77      2000



## Problem

The model detects only 19% of churned customers, indicating poor performance in identifying churn. 

Despite an accuracy of 81%, this is driven by a high recall of 97% for non-churned customers, showing a strong bias toward the majority class and weak detection of the minority (churn) class.

## Solution – Threshold Tuning and Probability-Based Predictions

Instead of relying on the default classification threshold (0.5), predicted probabilities are used to adjust the decision threshold.

Lowering the threshold enables the model to identify more potential churners, improving recall at the expense of precision.

This approach aligns with real-world scenarios where missing a churner is more costly than incorrectly flagging a non-churner.

In [28]:
y_prob = model.predict_proba(X_test)[:, 1]

y_pred_new = (y_prob > 0.25).astype(int)

print(classification_report(y_test, y_pred_new))

              precision    recall  f1-score   support

           0       0.89      0.80      0.84      1593
           1       0.43      0.60      0.50       407

    accuracy                           0.76      2000
   macro avg       0.66      0.70      0.67      2000
weighted avg       0.79      0.76      0.77      2000



Lowering the threshold from 0.5 to 0.25 increases churn recall from 19% to 60% (over 3x improvement), significantly improving the model’s ability to detect churners.

## Cost-Based Evaluation

To align model evaluation with business objectives, costs are assigned to prediction errors:

- False Negative (missed churner): High cost (lost customer)
- False Positive (incorrect churn prediction): Lower cost (unnecessary intervention)

By calculating total cost across different thresholds, the model can be evaluated based on its financial impact rather than just statistical metrics.

This approach enables selection of an optimal threshold that minimizes business loss.

In [29]:
#Testing different thresholds to see the movement of cost
thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]

for p in thresholds:
    y_pred = (y_prob > p).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    cost = fp * 100 + fn * 1000
    
    print(f"Threshold: {p}, Cost: {cost}")

Threshold: 0.05, Cost: 144300
Threshold: 0.1, Cost: 140200
Threshold: 0.15, Cost: 135000
Threshold: 0.2, Cost: 162500
Threshold: 0.25, Cost: 194400
Threshold: 0.3, Cost: 218300
Threshold: 0.35, Cost: 253900
Threshold: 0.4, Cost: 291200
Threshold: 0.45, Cost: 314600


In [30]:
# Analyze model performance across different thresholds to identify the optimal decision threshold

thresholds = [i/100 for i in range(1, 51)]  # 0.01 to 0.50
costs = []

for p in thresholds:
    y_pred = (y_prob > p).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    cost = fp * 100 + fn * 1000
    costs.append(cost)



best_idx = costs.index(min(costs))
best_threshold = thresholds[best_idx]
best_cost = costs[best_idx]


fig = go.Figure()

# Line plot
fig.add_trace(go.Scatter(
    x=thresholds,
    y=costs,
    mode='lines',
    name='Cost',
    line=dict(width=3, color='#17becf'),
    hovertemplate='Threshold: %{x:.2f}<br>Cost: $%{y:,}<extra></extra>'
))

# Highlight optimal point
fig.add_trace(go.Scatter(
    x=[best_threshold],
    y=[best_cost],
    mode='markers',
    name='Optimal Point',
    marker=dict(
        size=14,
        color='gray',
        line=dict(width=2, color='black')
    ),
    hovertemplate='Optimal Threshold: %{x:.2f}<br>Cost: $%{y:,}<extra></extra>'
))

# Annotation
fig.add_annotation(
    x=best_threshold,
    y=best_cost,
    text=f"<b>Optimal Threshold: </b>{best_threshold:.2f}<br>Cost: ${best_cost/1000:.1f}k",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40,
    bgcolor="gray",
    font=dict(color='white'),
    borderwidth=1,
    borderpad=4
)

# Vertical line
fig.add_vline(
    x=best_threshold,
    line_dash="dash",
    line_width=1,
    line_color="black"
)

# Horizontal line
fig.add_hline(
    y=best_cost,
    line_dash="dash",
    line_width=1,
    line_color="gray"
)

# Layout styling
fig.update_layout(
    title=dict(
        text="<b>Logistic Regression — Threshold Optimization</b>",
        x=0.5,
        xanchor='center',
        font=dict(size=24)
    ),
    xaxis_title=dict(
        text="<b>Threshold</b>",
        font=dict(size=18, color="gray")
    ),
    yaxis_title=dict(
        text="<b>Cost ($)</b>",
        font=dict(size=18, color="gray")
    ),
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=13),
    xaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    yaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

print(f"Best Threshold: {best_threshold:.3f}")
print(f"Minimum Cost: {best_cost:.2f}")

fig.show()

Best Threshold: 0.130
Minimum Cost: 133800.00


Threshold selection is based on minimizing total cost, with higher penalties assigned to missed churners (false negatives), prioritizing business impact over traditional metrics.

## Handling Class Imbalance

The dataset is imbalanced, with fewer churned customers compared to non-churned ones.

To address this, class weights are used to penalize misclassification of churned customers more heavily, encouraging the model to focus on the minority class.

In [31]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced')

In [32]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1593
           1       0.39      0.70      0.50       407

    accuracy                           0.71      2000
   macro avg       0.65      0.71      0.65      2000
weighted avg       0.80      0.71      0.74      2000



Applying class weighting shifts the model’s focus toward the minority class, significantly improving churn recall from 19% to 70% without threshold tuning. This demonstrates how model-level adjustments can address class imbalance, albeit with increased false positives.

In [33]:
# Analyze model performance across different thresholds to identify the optimal decision threshold

# Predict probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Threshold analysis
thresholds_lr = [i/100 for i in range(1, 51)]
costs_lr = []

for p in thresholds_lr:
    y_pred = (y_prob > p).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    cost = fp * 100 + fn * 1000
    costs_lr.append(cost)

# Best point
best_idx_lr = costs_lr.index(min(costs_lr))
best_threshold_lr = thresholds_lr[best_idx_lr]
best_cost_lr = costs_lr[best_idx_lr]

# Print results
print("Best Threshold:", round(best_threshold_lr, 3))
print("Minimum Cost:", best_cost_lr)


# -------------------------------
# Plot
# -------------------------------

fig = go.Figure()

# Line plot
fig.add_trace(go.Scatter(
    x=thresholds_lr,
    y=costs_lr,
    mode='lines',
    name='Cost',
    line=dict(width=3, color='#17becf'),
    hovertemplate='Threshold: %{x:.2f}<br>Cost: $%{y:,}<extra></extra>'
))

# Optimal point
fig.add_trace(go.Scatter(
    x=[best_threshold_lr],
    y=[best_cost_lr],
    mode='markers',
    name='Optimal Point',
    marker=dict(
        size=14,
        color='#1f77b4',
        line=dict(width=2, color='black')
    ),
    hovertemplate='Optimal Threshold: %{x:.2f}<br>Cost: $%{y:,}<extra></extra>'
))

# Annotation
fig.add_annotation(
    x=best_threshold_lr,
    y=best_cost_lr,
    text=f"<b>Optimal Threshold: </b>{best_threshold_lr:.2f}<br>Cost: ${best_cost_lr/1000:.1f}k",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40,
    bgcolor="#1f77b4",
    font=dict(color='white'),
    borderwidth=1,
    borderpad=4
)

# Reference lines
fig.add_vline(
    x=best_threshold_lr,
    line_dash="dash",
    line_width=1,
    line_color="black"
)

fig.add_hline(
    y=best_cost_lr,
    line_dash="dash",
    line_width=1,
    line_color="gray"
)

# Layout
fig.update_layout(
    title=dict(
        text="<b>Logistic Regression (Balanced) — Threshold Optimization</b>",
        x=0.5,
        xanchor='center',
        font=dict(size=24)
    ),
    xaxis_title=dict(
        text="<b>Threshold</b>",
        font=dict(size=18, color="gray")
    ),
    yaxis_title=dict(
        text="<b>Cost ($)</b>",
        font=dict(size=18, color="gray")
    ),
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=13),
    xaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    yaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

Best Threshold: 0.36
Minimum Cost: 132200


The class-weighted model reduces total cost from $133,800 to $132,200, achieving a net saving of $1,600. This highlights the value of prioritizing churn detection, as improvements in recall translate directly into reduced business loss.

---
## Random Forest Model

A Random Forest model is implemented to capture non-linear relationships between features.

Unlike Logistic Regression, Random Forest builds multiple decision trees and aggregates their predictions, allowing it to model more complex patterns in the data.

In [34]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.77      0.45      0.57       407

    accuracy                           0.86      2000
   macro avg       0.82      0.71      0.74      2000
weighted avg       0.85      0.86      0.85      2000



Random Forest improves churn recall to 45% (from 19% in Logistic Regression) while maintaining high precision (77%) and accuracy (86%). However, it still misses a significant portion of churned customers, indicating room for further improvement.

In [35]:
# Analyze model performance across different thresholds to identify the optimal decision threshold

# Predict probabilities
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# Threshold analysis
thresholds_rf = [i/100 for i in range(1, 51)]
costs_rf = []

for p in thresholds_rf:
    y_pred = (y_prob_rf > p).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    cost = fp * 100 + fn * 1000
    costs_rf.append(cost)

# Best point
best_idx_rf = costs_rf.index(min(costs_rf))
best_threshold_rf = thresholds_rf[best_idx_rf]
best_cost_rf = costs_rf[best_idx_rf]

# Print results
print("Best Threshold:", round(best_threshold_rf, 3))
print("Minimum Cost:", best_cost_rf)


# -------------------------------
# Plot
# -------------------------------

fig = go.Figure()

# Line plot
fig.add_trace(go.Scatter(
    x=thresholds_rf,
    y=costs_rf,
    mode='lines',
    name='Cost',
    line=dict(width=3, color='#17becf'),
    hovertemplate='Threshold: %{x:.2f}<br>Cost: $%{y:,}<extra></extra>'
))

# Optimal point
fig.add_trace(go.Scatter(
    x=[best_threshold_rf],
    y=[best_cost_rf],
    mode='markers',
    name='Optimal Point',
    marker=dict(
        size=14,
        color='#2ca02c',
        line=dict(width=2, color='black')
    ),
    hovertemplate='Optimal Threshold: %{x:.2f}<br>Cost: $%{y:,}<extra></extra>'
))

# Annotation
fig.add_annotation(
    x=best_threshold_rf,
    y=best_cost_rf,
    text=f"<b>Optimal Threshold: </b>{best_threshold_rf:.2f}<br>Cost: ${best_cost_rf/1000:.1f}k",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40,
    bgcolor="#2ca02c",
    font=dict(color='white'),
    borderwidth=1,
    borderpad=4
)

# Reference lines
fig.add_vline(
    x=best_threshold_rf,
    line_dash="dash",
    line_width=1,
    line_color="black"
)

fig.add_hline(
    y=best_cost_rf,
    line_dash="dash",
    line_width=1,
    line_color="gray"
)

# Layout
fig.update_layout(
    title=dict(
        text="<b>Random Forest — Threshold Optimization</b>",
        x=0.5,
        xanchor='center',
        font=dict(size=24)
    ),
    xaxis_title=dict(
        text="<b>Threshold</b>",
        font=dict(size=18, color="gray")
    ),
    yaxis_title=dict(
        text="<b>Cost ($)</b>",
        font=dict(size=18, color="gray")
    ),
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=13),
    xaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    yaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

Best Threshold: 0.07
Minimum Cost: 113700


The Random Forest model reduces total cost to $113,700, achieving a substantial improvement of over $20,000 compared to Logistic Regression. This demonstrates the effectiveness of capturing non-linear patterns for better churn detection and cost optimization.

---
## Gradient Boosting Model

Gradient Boosting is applied as an advanced ensemble technique where models are built sequentially.

Each new model focuses on correcting the errors of the previous ones, often resulting in improved predictive performance compared to other methods.

In [36]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(random_state=42)

gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_gb))

              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1593
           1       0.79      0.49      0.60       407

    accuracy                           0.87      2000
   macro avg       0.84      0.73      0.76      2000
weighted avg       0.86      0.87      0.86      2000



Gradient Boosting achieves the best performance so far, improving churn recall to 49% while maintaining high precision (79%) and accuracy (87%). However, it still misses a significant portion of churned customers.

In [37]:
# Analyze model performance across different thresholds to identify the optimal decision threshold

# Predict probabilities
y_prob_gb = gb_model.predict_proba(X_test)[:, 1]

# Threshold analysis
thresholds_gb = [i/100 for i in range(1, 51)]
costs_gb = []

for p in thresholds_gb:
    y_pred = (y_prob_gb > p).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    cost = fp * 100 + fn * 1000
    costs_gb.append(cost)

# Best point
best_idx_gb = costs_gb.index(min(costs_gb))
best_threshold_gb = thresholds_gb[best_idx_gb]
best_cost_gb = costs_gb[best_idx_gb]

# Print results
print("Best Threshold:", round(best_threshold_gb, 3))
print("Minimum Cost:", best_cost_gb)


# -------------------------------
# Plot
# -------------------------------

fig = go.Figure()

# Line plot
fig.add_trace(go.Scatter(
    x=thresholds_gb,
    y=costs_gb,
    mode='lines',
    name='Cost',
    line=dict(width=3, color='#17becf'),
    hovertemplate='Threshold: %{x:.2f}<br>Cost: $%{y:,}<extra></extra>'
))

# Optimal point
fig.add_trace(go.Scatter(
    x=[best_threshold_gb],
    y=[best_cost_gb],
    mode='markers',
    name='Optimal Point',
    marker=dict(
        size=14,
        color='#ff7f0e',
        line=dict(width=2, color='black')
    ),
    hovertemplate='Optimal Threshold: %{x:.2f}<br>Cost: $%{y:,}<extra></extra>'
))

# Annotation
fig.add_annotation(
    x=best_threshold_gb,
    y=best_cost_gb,
    text=f"<b>Optimal Threshold: </b>{best_threshold_gb:.2f}<br>Cost: ${best_cost_gb/1000:.1f}k",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40,
    bgcolor="#ff7f0e",
    font=dict(color='white'),
    borderwidth=1,
    borderpad=4
)

# Reference lines
fig.add_vline(
    x=best_threshold_gb,
    line_dash="dash",
    line_width=1,
    line_color="black"
)

fig.add_hline(
    y=best_cost_gb,
    line_dash="dash",
    line_width=1,
    line_color="gray"
)

# Layout
fig.update_layout(
    title=dict(
        text="<b>Gradient Boosting — Threshold Optimization</b>",
        x=0.5,
        xanchor='center',
        font=dict(size=24)
    ),
    xaxis_title=dict(
        text="<b>Threshold</b>",
        font=dict(size=18, color="gray")
    ),
    yaxis_title=dict(
        text="<b>Cost ($)</b>",
        font=dict(size=18, color="gray")
    ),
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=13),
    xaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    yaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

Best Threshold: 0.09
Minimum Cost: 104600


Gradient Boosting reduces total cost to $104,600, outperforming all previous models. This significant reduction demonstrates the effectiveness of sequential learning in capturing complex patterns and improving churn detection.

---
## XGBoost Model

XGBoost (Extreme Gradient Boosting) is an advanced ensemble technique that improves upon traditional gradient boosting by incorporating regularization and efficient computation.

It is widely used in industry due to its strong performance and ability to handle complex datasets effectively.

In this step, XGBoost is applied to further improve churn prediction performance and compare against previous models.

In [38]:
from xgboost import XGBClassifier

# -------------------------------
# Train Model
# -------------------------------
model_xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=4,
    eval_metric='logloss'
)

model_xgb.fit(X_train, y_train)

# -------------------------------
# Predict probabilities
# -------------------------------
y_prob_xgb = model_xgb.predict_proba(X_test)[:, 1]

# -------------------------------
# Threshold analysis - Analyze model performance across different thresholds to identify the optimal decision threshold
# -------------------------------
thresholds_xgb = [i/100 for i in range(1, 51)]
costs_xgb = []

for t in thresholds_xgb:
    y_pred = (y_prob_xgb > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    cost = fp * 100 + fn * 1000
    costs_xgb.append(cost)

# Best point
best_idx_xgb = costs_xgb.index(min(costs_xgb))
best_threshold_xgb = thresholds_xgb[best_idx_xgb]
best_cost_xgb = costs_xgb[best_idx_xgb]

# Confusion matrix at best threshold
tn, fp, fn, tp = confusion_matrix(
    y_test,
    (y_prob_xgb > best_threshold_xgb).astype(int)
).ravel()

# Print results
print("Best Threshold:", round(best_threshold_xgb, 3))
print("Minimum Cost:", best_cost_xgb)
print("FP:", fp, "| FN:", fn)
print()
print(classification_report(
    y_test,
    (y_prob_xgb > best_threshold_xgb).astype(int)
))


# -------------------------------
# Plot
# -------------------------------
fig = go.Figure()

# Line plot
fig.add_trace(go.Scatter(
    x=thresholds_xgb,
    y=costs_xgb,
    mode='lines',
    name='Cost',
    line=dict(width=3, color="#17becf"),
    hovertemplate='Threshold: %{x:.2f}<br>Cost: $%{y:,}<extra></extra>'
))

# Optimal point
fig.add_trace(go.Scatter(
    x=[best_threshold_xgb],
    y=[best_cost_xgb],
    mode='markers',
    name='Optimal Point',
    marker=dict(
        size=14,
        color='#d62728',
        line=dict(width=2, color='black')
    ),
    hovertemplate='Optimal Threshold: %{x:.2f}<br>Cost: $%{y:,}<extra></extra>'
))

# Annotation
fig.add_annotation(
    x=best_threshold_xgb,
    y=best_cost_xgb,
    text=f"<b>Optimal Threshold: </b>{best_threshold_xgb:.2f}<br>Cost: ${best_cost_xgb/1000:.1f}k",
    showarrow=True,
    arrowhead=2,
    ax=40,
    ay=-40,
    bgcolor="#d62728",
    font=dict(color='white'),
    borderwidth=1,
    borderpad=4
)

# Reference lines
fig.add_vline(
    x=best_threshold_xgb,
    line_dash="dash",
    line_width=1,
    line_color="black"
)

fig.add_hline(
    y=best_cost_xgb,
    line_dash="dash",
    line_width=1,
    line_color="gray"
)

# Layout
fig.update_layout(
    title=dict(
        text="<b>XGBoost — Threshold Optimization</b>",
        x=0.5,
        xanchor='center',
        font=dict(size=24)
    ),
    xaxis_title=dict(
        text="<b>Threshold</b>",
        font=dict(size=18, color="gray")
    ),
    yaxis_title=dict(
        text="<b>Cost ($)</b>",
        font=dict(size=18, color="gray")
    ),
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=13),
    xaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    yaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

Best Threshold: 0.23
Minimum Cost: 103500
FP: 755 | FN: 28

              precision    recall  f1-score   support

           0       0.97      0.53      0.68      1593
           1       0.33      0.93      0.49       407

    accuracy                           0.61      2000
   macro avg       0.65      0.73      0.59      2000
weighted avg       0.84      0.61      0.64      2000



---
## Model Comparison

The XGBoost model achieves the lowest cost of $103,500, outperforming all previous models including Gradient Boosting ($104,600), Random Forest ($113,700), and Logistic Regression ($133,800).

This represents a total cost reduction of over $30,000 compared to the baseline, making XGBoost the most effective model for minimizing business loss in churn prediction.

In [39]:

# Store all models' data
roc_models = {
    "Logistic Regression(Balanced)": {
        "thresholds": thresholds_lr,
        "costs": costs_lr,
        "best_threshold": best_threshold_lr,
        "best_cost": best_cost_lr,
        "color": "#1f77b4"
    },
    "Random Forest": {
        "thresholds": thresholds_rf,
        "costs": costs_rf,
        "best_threshold": best_threshold_rf,
        "best_cost": best_cost_rf,
        "color": "#2ca02c"
    },
    "Gradient Boosting": {
        "thresholds": thresholds_gb,
        "costs": costs_gb,
        "best_threshold": best_threshold_gb,
        "best_cost": best_cost_gb,
        "color": "#ff7f0e"
    },
    "XGBoost": {
        "thresholds": thresholds_xgb,
        "costs": costs_xgb,
        "best_threshold": best_threshold_xgb,
        "best_cost": best_cost_xgb,
        "color": "#d62728"
    }
}

fig = go.Figure()

# Plot all models
for name, data in roc_models.items():

    # Line
    fig.add_trace(go.Scatter(
        x=data["thresholds"],
        y=data["costs"],
        mode='lines',
        name=name,
        line=dict(width=3, color=data["color"]),
        hovertemplate=(
            f"{name}<br>"
            "Threshold: %{x:.2f}<br>"
            "Cost: $%{y:,}<extra></extra>"
        )
    ))

    # Optimal point
    fig.add_trace(go.Scatter(
        x=[data["best_threshold"]],
        y=[data["best_cost"]],
        mode='markers',
        showlegend=False,
        marker=dict(
            size=12,
            color=data["color"],
            line=dict(width=2, color='black')
        ),
        hovertemplate=(
            f"<b>{name} — Optimal</b><br>"
            "Threshold: %{x:.2f}<br>"
            "Cost: $%{y:,}<extra></extra>"
        )
    ))

# Layout
fig.update_layout(
    title=dict(
        text="<b>Threshold Optimization — Model Comparison</b>",
        x=0.5,
        xanchor='center',
        font=dict(size=24)
    ),
    xaxis_title=dict(
        text="<b>Threshold</b>",
        font=dict(size=18, color="gray")
    ),
    yaxis_title=dict(
        text="<b>Cost ($)</b>",
        font=dict(size=18, color="gray")
    ),
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=13),
    xaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    yaxis=dict(
        showline=True,
        linewidth=1,
        linecolor='black',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.05)'
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()


# -------------------------------
# Table: Threshold vs Cost
# -------------------------------

table_data = []

for name, data in roc_models.items():
    table_data.append([
        name,
        round(data["best_threshold"], 3),
        round(data["best_cost"], 2)
    ])

df_table = pd.DataFrame(
    table_data,
    columns=["Model", "Best Threshold", "Minimum Cost"]
)

# Sort by cost (ascending → best model first)
df_table = df_table.sort_values(by="Minimum Cost").reset_index(drop=True)

# Make index start from 1 instead of 0
df_table.index = df_table.index + 1
df_table.index.name = "Rank"

print("\nModel Comparison (Ranked by Cost):\n")
print(df_table)


Model Comparison (Ranked by Cost):

                              Model  Best Threshold  Minimum Cost
Rank                                                             
1                           XGBoost            0.23        103500
2                 Gradient Boosting            0.09        104600
3                     Random Forest            0.07        113700
4     Logistic Regression(Balanced)            0.36        132200


In [40]:
from sklearn.metrics import roc_auc_score, roc_curve

models = {
    "Logistic Regression": y_prob,
    "Random Forest": y_prob_rf,
    "Gradient Boosting": y_prob_gb,
    "XGBoost": y_prob_xgb
}

# Define custom colors (professional palette)
colors = {
    "Logistic Regression": "#1f77b4",   # blue
    "Random Forest": "#2ca02c",         # green
    "Gradient Boosting": "#ff7f0e",     # orange
    "XGBoost": "#d62728"                # red
}

fig = go.Figure()

# Plot ROC curves
for name, probs in models.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)

    fig.add_trace(go.Scatter(
        x=fpr,
        y=tpr,
        mode='lines',
        name=f"{name} (AUC = {auc:.3f})",
        line=dict(width=2, color=colors[name])
    ))

# Diagonal baseline
fig.add_trace(go.Scatter(
    x=[0,1],
    y=[0,1],
    mode='lines',
    line=dict(dash='dash', color='gray'),
    showlegend=False
))

# Layout
fig.update_layout(
    title=dict(
        text="<b>ROC Curve — Model Comparison</b>",
        x=0.5,
        font=dict(size=24)
    ),
    xaxis_title=dict(
        text="<b>False Positive Rate</b>",
        font=dict(size=18, color="gray")
    ),
    yaxis_title=dict(
        text="<b>True Positive Rate</b>",
        font=dict(size=18, color="gray")
    ),
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=13),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

## Business Recommendations

Based on the analysis, the following actions should be implemented to reduce customer churn:

- **Re-engage inactive customers:** Implement targeted re-engagement campaigns (notifications, offers, reminders) for inactive members, who show the highest churn risk.

- **Prioritize high-risk geographies:** Focus retention efforts in Germany by introducing localized offers and investigating region-specific drivers of churn.

- **Address product-related churn:** Investigate customers with three products to identify potential dissatisfaction or product misalignment, and introduce targeted retention incentives.

- **Protect high-value customers:** Proactively monitor customers with higher balances and offer premium support or personalized retention strategies.

- **Enable model-driven retention:** Deploy the model within business workflows to identify high-risk customers in real time and trigger targeted interventions, minimizing overall retention cost.

---

## Final Conclusion

This approach enables targeted, cost-efficient retention by focusing on high-risk customers rather than broad interventions.